In [37]:
import kagglehub
import os
import shutil
import pandas  as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from  sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import time 
import torch
import torch.nn  as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torchvision import transforms ,models
from PIL import Image
from transformers import BertTokenizer,BertModel
from torchvision.models import ResNet18_Weights


In [ ]:
df_styles = pd.read_csv('./fashion-dataset/styles.csv',on_bad_lines='skip')
df_images = pd.read_csv("./fashion-dataset/images.csv",on_bad_lines='skip')

In [ ]:
df_styles.head()

In [ ]:
df_images.head()

In [ ]:
print('df_styles',df_styles.columns)
print('df_images',df_images.columns)
print('df_styles',df_styles.shape)
print('df_images',df_images.shape)

In [ ]:
df_images['id'] = df_images['filename'].str.replace('.jpg','',regex =False)

In [ ]:
df_images

In [ ]:
df_images['id'] = df_images['id'].astype('int64')

In [ ]:
data  = pd.merge(df_styles , df_images,on='id',how='inner')
data.head()

In [ ]:
images_dir = 'fashion-dataset/images'
data['image_path'] = data['id'].apply(lambda x: os.path.join(images_dir, str(x) + ".jpg"))
data['image_path'] = data['image_path'].str.replace("\\",'/',regex =False)

data.info()

In [ ]:
data = data[data['image_path'].apply(os.path.exists)].reset_index(drop=True)
data.info()

In [ ]:
columns = ['link','id','year','masterCategory','articleType','filename']
data.drop(columns=columns,inplace=True)

In [ ]:
data['baseColour'] = data['baseColour'].fillna('Unknown')
data['season'] = data['season'].fillna('Unknown')
data['usage'] = data['usage'].fillna('Unknown')
data['productDisplayName'] = data['productDisplayName'].fillna('')

In [ ]:
data.info()

In [ ]:
class_counts = data['subCategory'].value_counts()
rare_classes = class_counts[class_counts<3].index.to_list()
data = data[~data['subCategory'].isin(rare_classes)].reset_index(drop=True)
data

In [ ]:
num_classes = data['subCategory'].nunique()
print(f"Number of classes: {num_classes}")

In [ ]:
data['subCategory'].value_counts()

In [ ]:
le = LabelEncoder()
data['label'] = le.fit_transform(data['subCategory'])
print(f"Sinif sayi  : {data['label'].nunique()}")

In [ ]:
train,test = train_test_split(data, test_size =0.2, random_state=42, stratify=data['label'] )
train,val  = train_test_split(train, test_size=0.2,random_state=42,stratify=train['label'])
print( train.shape)
print(test.shape)
print(val.shape)

In [ ]:
data['text'] = data['gender'] + " " + \
                data['baseColour'] + " " + \
                data['season'] + " " + \
                data['usage'] + " " + \
                data['productDisplayName']
data['text'].value_counts()

In [ ]:
train['text'] = train['gender'] + " " + train['baseColour'] + " " + train['season'] + " " + train['usage'] + " " + train['productDisplayName']
val['text'] = val['gender'] + " " + val['baseColour'] + " " + val['season'] + " " + val['usage'] + " " + val['productDisplayName']
test['text'] = test['gender'] + " " + test['baseColour'] + " " + test['season'] + " " + test['usage'] + " " + test['productDisplayName']


In [ ]:
class FashionDataset(Dataset):
    def __init__(self, df, tokenizer, transform=None, max_len=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_len = max_len
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        
        text = self.df.iloc[idx]['text']
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        label = torch.tensor(self.df.iloc[idx]['label'], dtype=torch.long)
        
        return {
            'image': image,
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': label
        }

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_dataset = FashionDataset(train, tokenizer, transform)
val_dataset = FashionDataset(val, tokenizer, transform)
test_dataset = FashionDataset(test, tokenizer, transform)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size )
test_loader = DataLoader(test_dataset, batch_size=batch_size )

In [ ]:
class MultiModalClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        self.cnn = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        self.cnn.fc = nn.Linear(512, 256)
        
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.text_fc = nn.Linear(768, 256)
        
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, image, input_ids, attention_mask):
        img_features = self.cnn(image)
        
        text_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        text_features = self.text_fc(text_output.pooler_output)
        
        fused = torch.cat([img_features, text_features], dim=1)
        
        return self.classifier(fused)
    


In [ ]:
print(torch.cuda.is_available()) 
print(torch.cuda.get_device_name(0))  

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiModalClassifier(num_classes).to(device)

for param in model.bert.parameters():
    param.requires_grad = False
    
class_weights = compute_class_weight('balanced', 
                                     classes=np.unique(train['label']), 
                                     y=train['label'])
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(),
                       lr=1e-4)

epochs = 2
for epoch in range(epochs):
    model.train()
    total_loss = 0
    start_time = time.time()
    
    for i, batch in enumerate(train_loader):
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        outputs = model(images, input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if i % 100 == 0:
            print(f"Batch {i}/{len(train_loader)}, Loss: {loss.item():.4f}")
    
    avg_loss = total_loss / len(train_loader)
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1} tamamlandı. Ort. Loss: {avg_loss:.4f}, Vaxt: {epoch_time:.1f} saniyə")
    
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label']
            
            outputs = model(images, input_ids, attention_mask)
            _, preds = torch.max(outputs, 1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.numpy())
    
    f1 = f1_score(val_labels, val_preds, average='macro')
    print(f'Epoch {epoch+1}: Val Macro F1 = {f1:.4f}\n')

In [ ]:
torch.save(model.state_dict(), 'multimodal_model.pth')

In [ ]:

model.eval()
test_preds, test_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label']
        
        outputs = model(images, input_ids, attention_mask)
        _, preds = torch.max(outputs, 1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.numpy())

 
print("Test Macro F1:", f1_score(test_labels, test_preds, average='macro'))
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=le.classes_))

In [ ]:
unique_test_labels = np.unique(test_labels)
test_class_names = le.inverse_transform(unique_test_labels)

print(f"Num classes in  Test  set: {len(unique_test_labels)}")
print(f" General class numbers {len(le.classes_)}")

print("Test Macro F1:", f1_score(test_labels, test_preds, average='macro'))
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, labels=unique_test_labels, target_names=test_class_names))